# Fine-tune the visit-it plan vectoriser

**Run this on Colab with a GPU** (Runtime → Change runtime type → T4 is enough).

## What this trains, and why

The pipeline currently reads floor plans with classical image processing: find the
darkest strokes, assume they are walls, watershed the space between them. That works
well on clean agency plans and fails on the ones it was never going to handle —
hand-drawn plans, unusual hatching, plans where the walls are not the darkest thing.

This notebook trains a small segmentation network to do the same job by having seen
thousands of plans instead of by rule. It outputs a checkpoint the pipeline loads as
the `learned` engine binding; the classical engine stays as the fallback and as the
baseline this has to beat.

**The pipeline works without this.** Nothing here is required to run visit-it — it is
the accuracy step, not the working step.

## What comes out

`plan_vectoriser.pt`, ~30 MB. Drop it in `models/` in the repo and stage 5 picks it up.

## Time and cost

| step | time on a free T4 |
|---|---|
| download CubiCasa5K (5.5 GB) | 10–20 min |
| build the training tensors | ~10 min |
| train 30 epochs | 60–90 min |
| **total** | **about 2 hours** |

Free Colab will disconnect before this finishes if you leave the tab. Checkpoints are
written every epoch to Google Drive so you can resume.


In [ ]:
#@title Check the GPU
import subprocess, sys
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout or 'NO GPU — set Runtime > Change runtime type')


In [ ]:
#@title Install
!pip install -q segmentation-models-pytorch==0.3.4 albumentations==1.4.14 opencv-python-headless
import torch, numpy as np, cv2, os, json, random
from pathlib import Path
torch.manual_seed(0); np.random.seed(0); random.seed(0)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch', torch.__version__, '| device', DEV)


In [ ]:
#@title Mount Drive so checkpoints survive a disconnect
from google.colab import drive
drive.mount('/content/drive')
WORK = Path('/content/drive/MyDrive/visit-it'); WORK.mkdir(parents=True, exist_ok=True)
print('checkpoints ->', WORK)


## 1 · Data

**CubiCasa5K** — 5,000 real floor plans, hand-annotated with room polygons and 80+
categories. It is the right corpus to start from because it is *raster* plans, which
is what we actually receive; ResPlan and Swiss Dwellings are vector and teach a model
about layouts rather than about images.

Zenodo record 2613548, ~5.5 GB. If the link has rotted, the same dataset is mirrored
on Kaggle as `cubicasa5k`.


In [ ]:
#@title Download CubiCasa5K
DATA = Path('/content/data'); DATA.mkdir(exist_ok=True)
if not (DATA/'cubicasa5k').exists():
    !wget -q --show-progress -O /content/data/cubicasa5k.zip \
        'https://zenodo.org/records/2613548/files/cubicasa5k.zip?download=1'
    !cd /content/data && unzip -q cubicasa5k.zip && rm cubicasa5k.zip
print(sorted(p.name for p in (DATA/'cubicasa5k').iterdir())[:10])


In [ ]:
#@title Build the training set
# Each sample: the plan image, and a mask with three classes —
#   0 outside the flat, 1 a room's interior, 2 a wall.
# That is exactly what stage 5 needs: rooms come out as connected components of
# class 1, and the walls tell it where the boundaries are.
import xml.etree.ElementTree as ET
SIZE = 512

def svg_to_mask(svg_path, w, h):
    """CubiCasa ships per-plan SVG annotations; rasterise rooms and walls."""
    tree = ET.parse(svg_path); root = tree.getroot()
    mask = np.zeros((h, w), np.uint8)
    ns = {'s': 'http://www.w3.org/2000/svg'}
    def polys(cls_contains):
        for g in root.iter():
            cls = (g.get('class') or '')
            if cls_contains not in cls: continue
            for poly in g.iter('{http://www.w3.org/2000/svg}polygon'):
                pts = poly.get('points','').strip().split()
                try: arr = np.array([[float(v) for v in p.split(',')] for p in pts])
                except ValueError: continue
                if len(arr) >= 3: yield arr.astype(np.int32)
    for arr in polys('Space'):   cv2.fillPoly(mask, [arr], 1)   # room interiors
    for arr in polys('Wall'):    cv2.fillPoly(mask, [arr], 2)   # walls on top
    return mask

def build_split(root_dir, out_npz, limit=None):
    xs, ys = [], []
    folders = sorted([p for p in Path(root_dir).rglob('model.svg')])
    if limit: folders = folders[:limit]
    for i, svg in enumerate(folders):
        img_p = svg.parent/'F1_scaled.png'
        if not img_p.exists(): img_p = svg.parent/'F1_original.png'
        if not img_p.exists(): continue
        img = cv2.imread(str(img_p), cv2.IMREAD_COLOR)
        if img is None: continue
        h, w = img.shape[:2]
        try: m = svg_to_mask(svg, w, h)
        except Exception: continue
        if m.max() == 0: continue
        xs.append(cv2.resize(img, (SIZE, SIZE), interpolation=cv2.INTER_AREA))
        ys.append(cv2.resize(m, (SIZE, SIZE), interpolation=cv2.INTER_NEAREST))
        if i % 250 == 0: print(f'{i}/{len(folders)}', flush=True)
    np.savez_compressed(out_npz, x=np.array(xs, np.uint8), y=np.array(ys, np.uint8))
    print(out_npz, np.array(xs).shape)

if not (WORK/'train.npz').exists():
    build_split(DATA/'cubicasa5k'/'high_quality_architectural', WORK/'train.npz')
if not (WORK/'val.npz').exists():
    build_split(DATA/'cubicasa5k'/'high_quality', WORK/'val.npz', limit=400)


## 2 · Train

A U-Net with a ResNet-34 encoder pretrained on ImageNet. Small, fast, and entirely
adequate for this: the task is finding regions bounded by lines, not recognising
objects. A RoomFormer-class transformer would predict polygons directly and is the
better long-term answer, but it needs far more GPU time and this gets the pipeline a
working learned engine today.

The loss weights walls more heavily than room interiors — walls are ~10% of the pixels
but they are the boundaries everything downstream depends on.


In [ ]:
#@title Datasets and augmentation
import albumentations as A
from torch.utils.data import Dataset, DataLoader

# Augmentations chosen for the failure modes we actually saw on real listings:
# colour-filled plans, faint scans, rotated phone photographs of a printed plan.
TRAIN_AUG = A.Compose([
    A.RandomRotate90(p=0.5), A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.15, rotate_limit=12, p=0.7),
    A.RandomBrightnessContrast(0.3, 0.3, p=0.7),
    A.HueSaturationValue(20, 40, 20, p=0.5),      # the colour-filled plans
    A.GaussNoise(p=0.3), A.ImageCompression(40, 90, p=0.3),
])

class Plans(Dataset):
    def __init__(self, npz, aug=None):
        d = np.load(npz); self.x, self.y, self.aug = d['x'], d['y'], aug
    def __len__(self): return len(self.x)
    def __getitem__(self, i):
        x, y = self.x[i], self.y[i]
        if self.aug:
            r = self.aug(image=x, mask=y); x, y = r['image'], r['mask']
        x = torch.from_numpy(x.transpose(2,0,1).astype(np.float32)/255.)
        return x, torch.from_numpy(y.astype(np.int64))

tr = DataLoader(Plans(WORK/'train.npz', TRAIN_AUG), batch_size=8, shuffle=True,
                num_workers=2, drop_last=True)
va = DataLoader(Plans(WORK/'val.npz'), batch_size=8, num_workers=2)
print(len(tr.dataset), 'train |', len(va.dataset), 'val')


In [ ]:
#@title Train
import segmentation_models_pytorch as smp

model = smp.Unet('resnet34', encoder_weights='imagenet', classes=3).to(DEV)
# Walls are ~10% of pixels and are the boundaries everything downstream needs.
w = torch.tensor([0.5, 1.0, 3.0], device=DEV)
loss_ce = torch.nn.CrossEntropyLoss(weight=w)
loss_dice = smp.losses.DiceLoss('multiclass')
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
EPOCHS = 30
sched = torch.optim.lr_scheduler.OneCycleLR(opt, 3e-4, epochs=EPOCHS, steps_per_epoch=len(tr))
scaler = torch.cuda.amp.GradScaler()

def room_iou(model, loader):
    model.eval(); inter = union = 0
    with torch.no_grad():
        for x, y in loader:
            p = model(x.to(DEV)).argmax(1).cpu()
            inter += ((p == 1) & (y == 1)).sum().item()
            union += ((p == 1) | (y == 1)).sum().item()
    return inter / max(union, 1)

best = 0.0
start = 0
ckpt = WORK/'plan_vectoriser_last.pt'
if ckpt.exists():
    st = torch.load(ckpt, map_location=DEV)
    model.load_state_dict(st['model']); opt.load_state_dict(st['opt'])
    start, best = st['epoch']+1, st['best']
    print(f'resumed from epoch {start}, best room IoU {best:.3f}')

for ep in range(start, EPOCHS):
    model.train(); tot = 0
    for x, y in tr:
        x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
        opt.zero_grad(set_to_none=True)
        with torch.cuda.amp.autocast():
            out = model(x); loss = loss_ce(out, y) + loss_dice(out, y)
        scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
        tot += loss.item()
    iou = room_iou(model, va)
    print(f'epoch {ep:2d}  loss {tot/len(tr):.4f}  room IoU {iou:.3f}', flush=True)
    torch.save({'model': model.state_dict(), 'opt': opt.state_dict(),
                'epoch': ep, 'best': max(best, iou)}, ckpt)
    if iou > best:
        best = iou
        torch.save({'model': model.state_dict(), 'room_iou': iou, 'size': SIZE,
                    'classes': ['outside','room','wall'], 'arch': 'unet_resnet34'},
                   WORK/'plan_vectoriser.pt')
        print('   saved new best')
print('done. best room IoU', round(best, 3))


## 3 · Check it on our own plans

CubiCasa is Finnish. Our listings are British. The number that matters is how it does
on **our** plans, not on the validation split — so upload a few and look at them.

Drag any plan image into Colab's file panel, or point this at a URL from
`data/golden/golden_set.json`.


In [ ]:
#@title Try it on a real UK listing plan
import urllib.request, matplotlib.pyplot as plt
URL = 'https://media.rightmove.co.uk/property-floorplan/35a312764/87977241/35a3127642a8ee16645bf099b27b77f4.png'  #@param {type:'string'}
urllib.request.urlretrieve(URL, '/content/test_plan.png')
img = cv2.imread('/content/test_plan.png', cv2.IMREAD_COLOR)
x = cv2.resize(img, (SIZE, SIZE), interpolation=cv2.INTER_AREA)
t = torch.from_numpy(x.transpose(2,0,1).astype(np.float32)/255.)[None].to(DEV)
model.eval()
with torch.no_grad(): pred = model(t).argmax(1)[0].cpu().numpy()
n, lab = cv2.connectedComponents((pred == 1).astype(np.uint8))
big = sum(1 for i in range(1, n) if (lab == i).sum() > SIZE*SIZE*0.005)
fig, ax = plt.subplots(1, 3, figsize=(15, 5))
ax[0].imshow(cv2.cvtColor(x, cv2.COLOR_BGR2RGB)); ax[0].set_title('plan')
ax[1].imshow(pred, cmap='viridis'); ax[1].set_title('outside / room / wall')
ax[2].imshow(lab, cmap='tab20'); ax[2].set_title(f'{big} rooms found')
for a in ax: a.axis('off')
plt.tight_layout(); plt.show()


## 4 · Take it back to the pipeline

Download `plan_vectoriser.pt` from Drive and put it in the repo:

```bash
mkdir -p models && mv ~/Downloads/plan_vectoriser.pt models/
python -m pipeline run 87977241 --only 5-plan     # picks it up automatically
python -m eval.harness --split dev --channel plan # did it beat the classical engine?
```

Stage 5 loads `models/plan_vectoriser.pt` if it is there and falls back to the
classical engine if it is not, so nothing breaks either way. The harness scores both
the same way, which is the point of having kept the classical engine as a baseline.

**Judge it on our plans, not on CubiCasa's validation split.** The target from
ROADMAP Sprint 3 is ≥80% room F1 on our own annotated plans. If it does not beat the
classical engine there, fine-tune on our 24 annotated UK plans — the format is in
`data/golden/annotations/` and `eval/annotations.py` describes it.


In [ ]:
#@title Download the checkpoint
from google.colab import files
p = WORK/'plan_vectoriser.pt'
print(p, f'{p.stat().st_size/1e6:.1f} MB' if p.exists() else 'not trained yet')
if p.exists(): files.download(str(p))
